### RAG with `ADK Agent` + `Knowledge Base Oracle 23ai vector db`

In [ ]:
import oci
import json
from oci.addons.adk import Agent, AgentClient, tool
from oci.object_storage import ObjectStorageClient
CONFIG_PROFILE = "DEFAULT"

### Tool Creation to retrive docs from Oracle 23ai

In [ ]:

@tool
def retrieve_documents(query: str) -> dict:
    """Retrieve course details from context"""
    import oracledb
    import oci
    from langchain_community.vectorstores.oraclevs import OracleVS
    from langchain_community.embeddings import OCIGenAIEmbeddings
    from langchain_community.vectorstores.utils import DistanceStrategy
    
    un = "vector"
    pw = "vector"
    cs = "localhost/FREEPDB1"
    
    import oracledb
    conn3c = oracledb.connect(user=un,password=pw,dsn=cs)
    
    embed_model = OCIGenAIEmbeddings(
    model_id="cohere.embed-english-v3.0",        
    service_endpoint="https://inference.generativeai.eu-frankfurt-1.oci.oraclecloud.com",        
    compartment_id="ocid1.compartment.oc1..aaaaaaaatvtprxtinkwbd4qoerdfoiay4huqrdfja2reqjmvpxrl353osbva",
    auth_type='INSTANCE_PRINCIPAL',)
    
    vs = OracleVS(embedding_function= embed_model,client=conn3c,
                                            table_name="DEMO_TABLE",
                                            distance_strategy=DistanceStrategy.DOT_PRODUCT)   
    
    retv = vs.as_retriever(search_type='similarity', search_kwargs={'k':3})
    
    content_doc = retv.get_relevant_documents(query)
    raw_text=''
    for i in range(len(content_doc)):
        raw_text += content_doc[i].page_content
    
    return {"content": raw_text}


### Agent initalization and Tool registration with Agent

In [ ]:
client = AgentClient(auth_type="api_key", profile=CONFIG_PROFILE, region="us-chicago-1")

agent = Agent(
    client=client,
        # Agent create on oci with name OCI-DEMO-AGENT-1
        agent_endpoint_id="ocid1.genaiagentendpoint.oc1.us-chicago-1.amaaaaaa2fm4ibaapplmaoebbhl2bkcp6ezpskriykxqmiwwxfhajbqjcayq",
    instructions="You are a smart assistant. Get information from context provided.",
    tools=[retrieve_documents]   # Tool Registration
)

agent.setup()

response = agent.run("what is oci ai foundation course") # Query1 To Ask

# response = agent.run("How many modules are in oci ai foundations course?")  # Query2 To Ask

print(response.data["message"]["content"]["text"])
